In [2]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances

df = load_all_snapshots(seasons=[2024])
print(df.shape)
print(sorted(pd.to_datetime(df["game_date"]).dt.year.unique()))

(710632, 119)
[np.int32(2024)]


In [3]:
from src.features.plate_discipline import discipline_profile
from src.features.batted_ball import quality_profile
from src.features.sample_size import STABILIZATION
from src.data.player_ids import load_player_ids, display_name

pa = plate_appearances(df)
woba = (
    pa.groupby("batter")
    .agg(woba_num=("woba_value", "sum"), woba_den=("woba_denom", "sum"))
)
woba["woba"] = pd.to_numeric(woba["woba_num"]) / pd.to_numeric(woba["woba_den"])

rows = []
for bid, g in df.groupby("batter"):
    if len(g) < 500:
        continue
    p = discipline_profile(g)
    p.update(quality_profile(g))
    p["batter"] = bid
    rows.append(p)

prof = pd.DataFrame(rows).set_index("batter").join(woba[["woba"]])

q = prof[
    (prof["n_out_of_zone"] >= STABILIZATION["chase_pct"][1])
    & (prof["n_zone_swings"] >= STABILIZATION["zone_contact_pct"][1])
    & (prof["bbe"] >= STABILIZATION["barrel_pct"][1])
    & prof["woba"].notna()
].copy()

ids = load_player_ids(q.index.tolist())
q = q.join(display_name(ids))

print(f"{len(q)} qualified batters")
print("Judge chase_pct:", round(q.loc[592450, "chase_pct"], 3), "(expect 0.179)")

343 qualified batters
Judge chase_pct: 0.179 (expect 0.179)


In [4]:
from sklearn.linear_model import LinearRegression

AXIS_METRICS = ["chase_pct", "zone_swing_pct", "zone_contact_pct", "barrel_pct"]
X = q[AXIS_METRICS].apply(lambda s: (s - s.mean()) / s.std())
y = q["woba"].astype(float)

lm = LinearRegression().fit(X, y)
coefs = pd.Series(lm.coef_, index=AXIS_METRICS).sort_values(key=abs, ascending=False)

print("standardised coefficients (wOBA per 1 sd):")
print(coefs.round(4).to_string())
print(f"R-squared: {lm.score(X, y):.4f}")
print()

q["score_learned"] = lm.predict(X)
print(q.nlargest(12, "score_learned")[
    ["name", "score_learned", "woba", "chase_pct", "zone_contact_pct", "barrel_pct"]
].round(3).to_string(index=False))

standardised coefficients (wOBA per 1 sd):
barrel_pct          0.0289
zone_contact_pct    0.0184
chase_pct          -0.0075
zone_swing_pct      0.0067
R-squared: 0.5011

              name  score_learned   woba  chase_pct  zone_contact_pct  barrel_pct
      Judge, Aaron          0.453  0.497      0.179             0.796       0.270
        Soto, Juan          0.415  0.432      0.178             0.855       0.198
    Ohtani, Shohei          0.410  0.447      0.259             0.803       0.218
     Seager, Corey          0.406  0.379      0.267             0.884       0.153
   Álvarez, Yordan          0.390  0.418      0.291             0.901       0.145
      Tucker, Kyle          0.389  0.425      0.171             0.887       0.129
  Carpenter, Kerry          0.385  0.399      0.326             0.832       0.178
       Witt, Bobby          0.381  0.424      0.315             0.877       0.143
Stanton, Giancarlo          0.381  0.341      0.309             0.779       0.209
Guerrero, 

In [5]:
from sklearn.model_selection import KFold
from scipy.stats import spearmanr

# Do learned weights survive resampling?
kf = KFold(n_splits=5, shuffle=True, random_state=42)
coef_runs = []
for train_idx, _ in kf.split(X):
    m = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    coef_runs.append(m.coef_)

coef_df = pd.DataFrame(coef_runs, columns=AXIS_METRICS)
print("coefficient stability across 5 folds:")
print(coef_df.agg(["mean", "std", "min", "max"]).round(4).to_string())
print()

# Rank stability
rank_runs = []
for train_idx, _ in kf.split(X):
    m = LinearRegression().fit(X.iloc[train_idx], y.iloc[train_idx])
    rank_runs.append(pd.Series(m.predict(X), index=q.index).rank(ascending=False))

ranks = pd.DataFrame(rank_runs).T
print("rank correlation between folds:")
print(ranks.corr(method="spearman").round(3).to_string())

coefficient stability across 5 folds:
      chase_pct  zone_swing_pct  zone_contact_pct  barrel_pct
mean    -0.0075          0.0066            0.0184      0.0289
std      0.0013          0.0009            0.0005      0.0011
min     -0.0091          0.0053            0.0177      0.0277
max     -0.0060          0.0074            0.0190      0.0304

rank correlation between folds:
       0      1      2      3      4
0  1.000  0.995  0.990  0.992  0.998
1  0.995  1.000  0.997  0.996  0.997
2  0.990  0.997  1.000  0.999  0.995
3  0.992  0.996  0.999  1.000  0.995
4  0.998  0.997  0.995  0.995  1.000
